# Individual Task 1 - Part 1.3: Data Analysis

Two publicly available healthcare datasets, two machine learning algorithms each.

* **Dataset A** - Diabetes 130-US Hospitals (1999-2008), UCI id=296
* **Dataset B** - Heart Disease, Cleveland, UCI id=45

* **Model 1** - Logistic Regression (regularised, class-weighted)
* **Model 2** - Histogram Gradient Boosting

Neither model was used in COSC2670/2738 (which used item-based collaborative
filtering, decision trees, kNN and Random Forest), satisfying the requirement
that at least one algorithm differs from previous coursework.

Run order: cells top to bottom. Requires internet on first run to pull the
datasets from UCI; after that they are cached to ./data/.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

In [ ]:
!pip install ucimlrepo

## 1. Data loading

`ucimlrepo` returns a dataset in three blocks: `features`, `targets`, and
sometimes `ids`. For the diabetes data, `encounter_id` and `patient_nbr` are
in `ids`. Joining only features and targets drops them, and the deduplication
step in section 2 then finds nothing to deduplicate and skips without
complaint. All three blocks are joined below, and an assertion checks the
columns arrived.

**If you have a cached `data/diabetes_130us.csv` from an earlier run, delete
it first** -- the cache is read before the download and will reproduce the bug.


In [ ]:
def load_uci(dataset_id, cache_name):
    """Fetch a UCI dataset, caching to CSV so later runs are offline.

    ucimlrepo splits a dataset into features / targets / ids. The diabetes data
    puts encounter_id and patient_nbr in `ids`, so joining only features and
    targets silently drops them -- which disables the deduplication that
    prevents one patient appearing in both train and test partitions. All three
    blocks are joined here.
    """
    cache = os.path.join("data", f"{cache_name}.csv")
    if os.path.exists(cache):
        df = pd.read_csv(cache, low_memory=False)
        print(f"Loaded cached {cache_name}: {df.shape}")
        return df

    from ucimlrepo import fetch_ucirepo  # pip install ucimlrepo
    repo = fetch_ucirepo(id=dataset_id)

    parts = []
    for attr in ("ids", "features", "targets"):
        block = getattr(repo.data, attr, None)
        if block is not None and len(getattr(block, "columns", [])):
            parts.append(block)
    df = pd.concat(parts, axis=1)
    df = df.loc[:, ~df.columns.duplicated()]

    df.to_csv(cache, index=False)
    print(f"Downloaded {cache_name}: {df.shape}")
    return df


diabetes_raw = load_uci(296, "diabetes_130us")
heart_raw = load_uci(45, "heart_cleveland")

print("Diabetes:", diabetes_raw.shape)
print("Heart:   ", heart_raw.shape)

# Fail loudly rather than silently skipping a cleaning step.
required = ["patient_nbr", "discharge_disposition_id", "readmitted", "weight"]
absent = [c for c in required if c not in diabetes_raw.columns]
assert not absent, (
    f"Missing columns: {absent}. Delete data/diabetes_130us.csv and re-run -- "
    "a stale cache from an earlier version is the usual cause."
)
print(f"\nColumn check passed. {diabetes_raw['patient_nbr'].nunique():,} unique "
      f"patients across {len(diabetes_raw):,} encounters.")

## 2. Dataset A - Diabetes 130-US Hospitals

### 2.1 Cleaning decisions

Every step below is a judgement call worth defending in the report:

| Step | Reason |
|---|---|
| Missing values are encoded `?`, not `NaN` | Must be converted or every column reads as object |
| Drop `weight` | ~97% missing - imputation would be fabrication |
| Keep `payer_code` / `medical_specialty` as an explicit "Unknown" level | Missingness is itself informative (who records specialty?) |
| Drop encounters ending in death or hospice (discharge codes 11, 13, 14, 19, 20, 21) | These patients cannot be readmitted - including them injects label noise |
| Keep only the first encounter per `patient_nbr` | Repeat encounters from one patient leak across the train/test split |
| Drop `examide`, `citoglipton` | Single-valued, zero variance |
| Group ICD-9 `diag_1` into clinical chapters | 700+ raw codes would explode one-hot dimensionality |

Target: `readmitted == '<30'` -> 1, else 0.

In [ ]:
def group_icd9(code):
    """Collapse an ICD-9 code into a clinical chapter (Strack et al., 2014)."""
    if pd.isna(code):
        return "Missing"
    s = str(code)
    if s.startswith("V") or s.startswith("E"):
        return "Other"
    try:
        v = float(s)
    except ValueError:
        return "Other"
    if 390 <= v < 460 or int(v) == 785:
        return "Circulatory"
    if 460 <= v < 520 or int(v) == 786:
        return "Respiratory"
    if 520 <= v < 580 or int(v) == 787:
        return "Digestive"
    if int(v) == 250:
        return "Diabetes"
    if 800 <= v < 1000:
        return "Injury"
    if 710 <= v < 740:
        return "Musculoskeletal"
    if 580 <= v < 630 or int(v) == 788:
        return "Genitourinary"
    if 140 <= v < 240:
        return "Neoplasms"
    return "Other"


def prepare_diabetes(df):
    df = df.replace("?", np.nan).copy()

    # 1. Remove encounters that cannot produce a readmission
    dead_or_hospice = [11, 13, 14, 19, 20, 21]
    if "discharge_disposition_id" in df.columns:
        df = df[~df["discharge_disposition_id"].isin(dead_or_hospice)]

    # 2. One row per patient - prevents the same patient appearing in train and test
    if "patient_nbr" in df.columns:
        df = df.drop_duplicates(subset="patient_nbr", keep="first")

    # 3. Target
    df["target"] = (df["readmitted"] == "<30").astype(int)

    # 4. Feature engineering
    for c in ["diag_1", "diag_2", "diag_3"]:
        if c in df.columns:
            df[c + "_grp"] = df[c].apply(group_icd9)
    df["prior_visits"] = (
        df.get("number_outpatient", 0)
        + df.get("number_emergency", 0)
        + df.get("number_inpatient", 0)
    )

    # 5. Drop
    drop = ["encounter_id", "patient_nbr", "weight", "readmitted",
            "examide", "citoglipton", "diag_1", "diag_2", "diag_3"]
    df = df.drop(columns=[c for c in drop if c in df.columns])

    # 6. Admin ID codes are categorical, not ordinal integers
    for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
        if c in df.columns:
            df[c] = df[c].astype(str)

    return df.reset_index(drop=True)


dia = prepare_diabetes(diabetes_raw)
y_dia = dia.pop("target")
X_dia = dia

assert len(X_dia) < 0.85 * len(diabetes_raw), (
    f"Only {len(diabetes_raw) - len(X_dia):,} rows removed. Deduplication "
    "probably did not run -- check that patient_nbr survived loading."
)

print(f"After cleaning: {X_dia.shape[0]:,} encounters, {X_dia.shape[1]} features")
print(f"Removed {len(diabetes_raw) - len(X_dia):,} rows "
      f"(death/hospice discharges, then repeat encounters per patient)")
print(f"Positive class (readmitted <30 days): {y_dia.mean():.1%}")
print("\nA classifier that always predicts 'no readmission' scores "
      f"{1 - y_dia.mean():.1%} accuracy. This is why accuracy is not the "
      "headline metric for this dataset.")

### 2.2 Preprocessing pipeline

`age` arrives as brackets (`[0-10)`, `[10-20)`, ...) so it is ordinal, not
nominal. Everything else numeric is scaled (needed for logistic regression,
harmless for the boosted trees).

In [ ]:
AGE_ORDER = [["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
              "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]]

num_cols_dia = X_dia.select_dtypes(include=np.number).columns.tolist()
cat_cols_dia = [c for c in X_dia.columns if c not in num_cols_dia and c != "age"]

pre_dia = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols_dia),
    ("age", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ord", OrdinalEncoder(categories=AGE_ORDER,
                                             handle_unknown="use_encoded_value",
                                             unknown_value=-1)),
                      ("sc", StandardScaler())]), ["age"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("oh", OneHotEncoder(handle_unknown="ignore",
                                           min_frequency=30,
                                           sparse_output=False))]), cat_cols_dia),
], remainder="drop")

Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(
    X_dia, y_dia, test_size=0.25, stratify=y_dia, random_state=RANDOM_STATE)
print(f"Train {Xtr_d.shape[0]:,} / Test {Xte_d.shape[0]:,}")

### 2.3 Models

`class_weight='balanced'` on the logistic regression, and a manually computed
equivalent for the booster, so that neither model simply learns the majority
class.

In [ ]:
pos_weight = (ytr_d == 0).sum() / (ytr_d == 1).sum()

logreg_dia = Pipeline([
    ("pre", pre_dia),
    ("clf", LogisticRegression(max_iter=2000, C=0.1, penalty="l2",
                               class_weight="balanced",
                               solver="lbfgs", random_state=RANDOM_STATE)),
])

hgb_dia = Pipeline([
    ("pre", pre_dia),
    ("clf", HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.06, max_leaf_nodes=31,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.15,
        class_weight="balanced", random_state=RANDOM_STATE)),
])

models_dia = {"Logistic Regression": logreg_dia, "Gradient Boosting": hgb_dia}
for name, m in models_dia.items():
    print(f"Fitting {name} ...")
    m.fit(Xtr_d, ytr_d)
print("Done.")

## 3. Evaluation helpers

`precision_at_k` is the operationally honest metric for a PHN: the
care-coordination team can only call a fixed number of patients per month, so
what matters is the hit-rate inside the top decile of predicted risk, not
global accuracy.

In [ ]:
def precision_at_k(y_true, y_score, k_frac=0.10):
    n = max(1, int(len(y_score) * k_frac))
    idx = np.argsort(y_score)[::-1][:n]
    return np.asarray(y_true)[idx].mean()


def recall_at_k(y_true, y_score, k_frac=0.10):
    y_true = np.asarray(y_true)
    n = max(1, int(len(y_score) * k_frac))
    idx = np.argsort(y_score)[::-1][:n]
    return y_true[idx].sum() / max(1, y_true.sum())


def evaluate(name, model, X_test, y_test, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced acc": balanced_accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall (sens.)": recall_score(y_test, pred, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "Brier": brier_score_loss(y_test, proba),
        "Prec@10%": precision_at_k(y_test, proba, 0.10),
        "Recall@10%": recall_at_k(y_test, proba, 0.10),
    }, proba, pred

In [ ]:
rows_d, probas_d = [], {}
for name, m in models_dia.items():
    row, proba, pred = evaluate(name, m, Xte_d, yte_d)
    rows_d.append(row)
    probas_d[name] = proba
    print(f"\n--- {name} (Diabetes) ---")
    print(classification_report(yte_d, pred, target_names=["Not readmitted", "Readmitted <30d"],
                                digits=3, zero_division=0))
    print("Confusion matrix (rows = actual):")
    print(pd.DataFrame(confusion_matrix(yte_d, pred),
                       index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))

results_dia = pd.DataFrame(rows_d).set_index("Model").round(3)
print("\n=== Dataset A: Diabetes 130-US Hospitals ===")
print(results_dia.T)

baseline = pd.Series({
    "Accuracy": 1 - yte_d.mean(), "Balanced acc": 0.5, "ROC-AUC": 0.5,
    "PR-AUC": yte_d.mean(), "Prec@10%": yte_d.mean(),
}, name="Majority-class baseline")
print("\nBaseline for reference:")
print(baseline.round(3))

## 4. Dataset B - Heart Disease (Cleveland)

303 records. With n this small a single train/test split is unstable, so the
headline numbers come from repeated stratified 10-fold cross-validation and
are reported with a standard deviation.

In [ ]:
def prepare_heart(df):
    df = df.replace("?", np.nan).copy()
    target_col = "num" if "num" in df.columns else df.columns[-1]
    df["target"] = (pd.to_numeric(df[target_col], errors="coerce") > 0).astype(int)
    df = df.drop(columns=[target_col])
    for c in df.columns:
        if c != "target":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.reset_index(drop=True)


hrt = prepare_heart(heart_raw)
y_hrt = hrt.pop("target")
X_hrt = hrt

print(f"{X_hrt.shape[0]} patients, {X_hrt.shape[1]} attributes")
print(f"Positive class (disease present): {y_hrt.mean():.1%}")
print("Missing values per column:")
print(X_hrt.isna().sum()[X_hrt.isna().sum() > 0])

HEART_NOMINAL = [c for c in ["cp", "restecg", "slope", "thal"] if c in X_hrt.columns]
HEART_NUMERIC = [c for c in X_hrt.columns if c not in HEART_NOMINAL]

pre_hrt = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), HEART_NUMERIC),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore",
                                           sparse_output=False))]), HEART_NOMINAL),
])

logreg_hrt = Pipeline([("pre", pre_hrt),
                       ("clf", LogisticRegression(max_iter=2000, C=1.0,
                                                  class_weight="balanced",
                                                  random_state=RANDOM_STATE))])
hgb_hrt = Pipeline([("pre", pre_hrt),
                    ("clf", HistGradientBoostingClassifier(
                        max_iter=200, learning_rate=0.05, max_leaf_nodes=8,
                        min_samples_leaf=10, l2_regularization=1.0,
                        random_state=RANDOM_STATE))])

models_hrt = {"Logistic Regression": logreg_hrt, "Gradient Boosting": hgb_hrt}

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy",
           "recall": "recall", "precision": "precision", "f1": "f1",
           "roc_auc": "roc_auc", "average_precision": "average_precision"}

cv_rows = []
for name, m in models_hrt.items():
    cvres = cross_validate(m, X_hrt, y_hrt, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"Model": name}
    for k in scoring:
        row[k] = f"{cvres['test_' + k].mean():.3f} ± {cvres['test_' + k].std():.3f}"
    cv_rows.append(row)

results_hrt = pd.DataFrame(cv_rows).set_index("Model")
print("\n=== Dataset B: Heart Disease (Cleveland), 10-fold stratified CV ===")
print(results_hrt.T)

# Single hold-out fit for curves and coefficient inspection
Xtr_h, Xte_h, ytr_h, yte_h = train_test_split(
    X_hrt, y_hrt, test_size=0.25, stratify=y_hrt, random_state=RANDOM_STATE)
probas_h = {}
for name, m in models_hrt.items():
    m.fit(Xtr_h, ytr_h)
    probas_h[name] = m.predict_proba(Xte_h)[:, 1]
    print(f"\n--- {name} (Heart, hold-out) ---")
    print(classification_report(yte_h, m.predict(Xte_h),
                                target_names=["No disease", "Disease"],
                                digits=3, zero_division=0))

## 5. Figures

Three figures per dataset: ROC, precision-recall, calibration. The PR curve
is the informative one for the imbalanced diabetes data; ROC flatters
imbalanced classifiers.

In [ ]:
def plot_curves(y_true, proba_dict, title, fname):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

    for name, p in proba_dict.items():
        fpr, tpr, _ = roc_curve(y_true, p)
        ax[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(y_true, p):.3f})")
    ax[0].plot([0, 1], [0, 1], "k--", lw=1)
    ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC")
    ax[0].legend(fontsize=8)

    for name, p in proba_dict.items():
        pr, rc, _ = precision_recall_curve(y_true, p)
        ax[1].plot(rc, pr, lw=2, label=f"{name} (AP={average_precision_score(y_true, p):.3f})")
    ax[1].axhline(np.mean(y_true), color="k", ls="--", lw=1, label="Baseline (prevalence)")
    ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
    ax[1].legend(fontsize=8)

    for name, p in proba_dict.items():
        frac, mean_pred = calibration_curve(y_true, p, n_bins=10, strategy="quantile")
        ax[2].plot(mean_pred, frac, "o-", lw=2, label=name)
    ax[2].plot([0, 1], [0, 1], "k--", lw=1)
    ax[2].set(xlabel="Mean predicted probability", ylabel="Observed frequency",
              title="Calibration")
    ax[2].legend(fontsize=8)

    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(f"figures/{fname}", dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figures/{fname}")


plot_curves(yte_d, probas_d, "Dataset A - 30-day readmission", "diabetes_curves.png")
plot_curves(yte_h, probas_h, "Dataset B - Heart disease presence", "heart_curves.png")

## 6. What is actually driving the predictions

Two views: signed logistic-regression coefficients (direction and magnitude,
interpretable to a clinician) and permutation importance for the booster
(model-agnostic, captures non-linear contributions).

In [ ]:
def logreg_coefficients(pipe, top=15):
    names = pipe.named_steps["pre"].get_feature_names_out()
    coefs = pipe.named_steps["clf"].coef_[0]
    s = pd.Series(coefs, index=names)
    return pd.concat([s.nlargest(top), s.nsmallest(top)]).sort_values(ascending=False)


print("\n=== Diabetes: logistic regression coefficients (log-odds) ===")
print(logreg_coefficients(models_dia["Logistic Regression"]).round(3))

print("\n=== Heart: logistic regression coefficients (log-odds) ===")
print(logreg_coefficients(models_hrt["Logistic Regression"], top=10).round(3))

# Permutation importance on a subsample (full 25k x 300 permutations is slow)
sub = min(6000, len(Xte_d))
perm = permutation_importance(
    models_dia["Gradient Boosting"], Xte_d.iloc[:sub], yte_d.iloc[:sub],
    scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp_d = (pd.Series(perm.importances_mean, index=Xte_d.columns)
         .sort_values(ascending=False).head(15))
print("\n=== Diabetes: gradient boosting permutation importance (drop in PR-AUC) ===")
print(imp_d.round(5))

perm_h = permutation_importance(
    models_hrt["Gradient Boosting"], Xte_h, yte_h,
    scoring="roc_auc", n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp_h = (pd.Series(perm_h.importances_mean, index=Xte_h.columns)
         .sort_values(ascending=False))
print("\n=== Heart: gradient boosting permutation importance (drop in ROC-AUC) ===")
print(imp_h.round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
imp_d.head(12).iloc[::-1].plot.barh(ax=ax[0], color="#4c72b0")
ax[0].set(title="Diabetes - permutation importance", xlabel="Drop in PR-AUC")
imp_h.head(12).iloc[::-1].plot.barh(ax=ax[1], color="#dd8452")
ax[1].set(title="Heart - permutation importance", xlabel="Drop in ROC-AUC")
fig.tight_layout()
fig.savefig("figures/feature_importance.png", dpi=160, bbox_inches="tight")
plt.close(fig)
print("Saved figures/feature_importance.png")

## 7. Threshold selection

The 0.5 cut-off is arbitrary. Below, the threshold is chosen to reflect the
real asymmetry: a missed readmission costs far more than an unnecessary
follow-up phone call. The table lets you state a defensible operating point
in the report rather than accepting the default.

In [ ]:
def threshold_table(y_true, proba, thresholds=(0.30, 0.40, 0.50, 0.60, 0.70)):
    out = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        out.append({
            "Threshold": t,
            "Flagged": int(pred.sum()),
            "TP": tp, "FP": fp, "FN": fn,
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        })
    return pd.DataFrame(out).round(3)


print("\n=== Diabetes: gradient boosting, threshold sensitivity ===")
print(threshold_table(yte_d, probas_d["Gradient Boosting"]).to_string(index=False))

print("\n=== Heart: logistic regression, threshold sensitivity ===")
print(threshold_table(yte_h, probas_h["Logistic Regression"]).to_string(index=False))

## 8. Export tables for the LaTeX write-up

In [ ]:
results_dia.to_csv("figures/results_diabetes.csv")
results_hrt.to_csv("figures/results_heart.csv")

print("\n--- LaTeX: Dataset A ---")
print(results_dia.to_latex(float_format="%.3f",
                           caption="Test-set performance, 30-day readmission prediction.",
                           label="tab:diabetes"))
print("\n--- LaTeX: Dataset B ---")
print(results_hrt.to_latex(caption="10-fold cross-validated performance, heart disease.",
                           label="tab:heart"))

print("\nAll outputs written to ./figures/")